# ECLIPSE · Notebook 03 — Evaluation

Comprehensive evaluation of a trained ECLIPSE-PRIME model:
- 4-class confusion matrix
- Per-class F1, AUC-ROC, balanced accuracy
- Transit parameter RMSE / MAE (TRANSIT-class only)
- Reliability diagrams (calibration)
- Injection-recovery sensitivity curve

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.eclipse_prime import ECLIPSEPrime
from src.utils.config import DEFAULT_CONFIG, CLASS_NAMES
from src.utils.checkpoint import get_best_checkpoint, load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ECLIPSEPrime.from_config(DEFAULT_CONFIG).to(device)

ckpt = get_best_checkpoint(DEFAULT_CONFIG.api.checkpoint_dir)
if ckpt:
    load_checkpoint(ckpt, model, device=device)
    print(f'Loaded checkpoint: {ckpt}')
else:
    print('No checkpoint found. Run notebook 02 first.')

model.eval()
print('Model ready for evaluation.')

In [ ]:
# Load test set
from pathlib import Path
from src.training.data_loader import make_eclipse_dataloader

test_csv = '../data/processed/tce_catalog_test.csv'
if not Path(test_csv).exists():
    print(f'{test_csv} not found. Using val set.')
    test_csv = '../data/processed/tce_catalog_val.csv'

if Path(test_csv).exists():
    test_loader = make_eclipse_dataloader(test_csv, split='test', augment=False, batch_size=32)
    print(f'Test set: {len(test_loader.dataset)} samples')
else:
    print('No test CSV available.')

In [ ]:
# Collect predictions
from pathlib import Path

if Path(test_csv).exists():
    all_labels, all_preds, all_probs = [], [], []
    all_true_params, all_pred_params = [], []
    all_transit = []

    with torch.no_grad():
        for batch in test_loader:
            out = model(
                batch['raw_flux'].to(device),
                batch['global_view'].to(device),
                batch['local_view'].to(device),
                batch['stellar'].to(device),
                batch['centroid'].to(device)
            )
            probs = out['probs'].cpu().numpy()
            all_labels.extend(batch['class'].numpy().tolist())
            all_preds.extend(np.argmax(probs, axis=1).tolist())
            all_probs.append(probs)
            all_transit.extend(batch['has_params'].numpy().tolist())
            all_true_params.append(np.stack([
                batch['period'].numpy(), batch['duration'].numpy(), batch['depth'].numpy()
            ], axis=1))
            all_pred_params.append(np.stack([
                out['period_mean'].cpu().numpy(), out['duration_mean'].cpu().numpy(), out['depth_mean'].cpu().numpy()
            ], axis=1))

    y_true  = np.array(all_labels)
    y_pred  = np.array(all_preds)
    y_probs = np.concatenate(all_probs, axis=0)
    y_params_true = np.concatenate(all_true_params, axis=0)
    y_params_pred = np.concatenate(all_pred_params, axis=0)
    transit_mask = np.array(all_transit, dtype=bool)
    print(f'Collected {len(y_true)} predictions.')

In [ ]:
# Compute all metrics
from src.training.metrics import compute_all_metrics

if Path(test_csv).exists():
    metrics = compute_all_metrics(
        y_true_cls=y_true, y_pred_cls=y_pred, y_probs=y_probs,
        y_true_params=y_params_true, y_pred_params=y_params_pred,
        transit_mask=transit_mask
    )

    print('=== ECLIPSE-PRIME Evaluation Results ===')
    print(f'Macro F1:         {metrics["f1_macro"]:.4f}')
    print(f'AUC-ROC (macro):  {metrics["auc_macro"]:.4f}')
    print(f'Balanced Acc:     {metrics["balanced_acc"]:.4f}')
    print()
    print('Per-class F1:')
    for cls in ['transit', 'eb', 'blend', 'other']:
        print(f'  {cls.upper():8s}: {metrics.get(f"f1_{cls}", 0):.4f}')
    print()
    print('Parameter estimation (TRANSIT only):')
    for param in ['period', 'duration', 'depth']:
        print(f'  {param.upper():10s}: RMSE={metrics.get(f"{param}_rmse", float("nan")):.4f}, MAE={metrics.get(f"{param}_mae", float("nan")):.4f}')

In [ ]:
# 4-class confusion matrix
from src.evaluation.evaluation import plot_4class_confusion_matrix, plot_reliability_diagram

if Path(test_csv).exists():
    cm_path = plot_4class_confusion_matrix(y_true, y_pred, output_path='../data/reports/confusion_matrix.png')
    from IPython.display import Image
    Image(cm_path)

In [ ]:
# Reliability diagrams
if Path(test_csv).exists():
    rel_path = plot_reliability_diagram(y_true, y_probs, output_path='../data/reports/reliability_diagram.png')
    from IPython.display import Image
    Image(rel_path)